Linear Mixed Model Analysis

In [2]:
### Set Paths for Statistical Analysis
### Set Paths for Statistical Analysis
visgam_dir  <- dirname(getwd())
results_dir <- file.path(visgam_dir, "Results", "SNR_Controls", "Revision_IN")
### ----- Path to dataset ----- ###
dataset     <- file.path(results_dir, "results_itpc", "itpc 10_20.csv")
var_itpc    <- 'itpc_high'
### Load Packages 
library(tidyr)
library(dplyr)

In [3]:
##### Long Format CSV
### ----- Read Dataset 
df      <- read.csv(dataset)

### ----- Transform to Long Format
df_long <- df %>%
  pivot_longer(
    cols          = c(itpc_high_eeg, itpc_low_eeg, itpc_high_opm, itpc_low_opm),
    names_to      = c("Measure", "Modality"),
    names_pattern = "^(itpc_(?:high|low))_(eeg|opm)$", 
    values_to     = "Value"
  ) %>%
  mutate(
    Modality      = toupper(Modality), # Quickly turns eeg/opm to EEG/OPM
    SZ_EEG_INFO   = factor(SZ_EEG_INFO, levels=c(0,1), labels=c("Control","Patient"))
  ) %>%
  filter(SZ_EEG_INFO == "Control")
  
df_model <- df_long %>%
  filter(Measure == var_itpc) %>%
  mutate(Modality = factor(Modality))

In [4]:
### Load Packages for Statistical Analysis
library(lme4)
library(MuMIn)
library(lmerTest)
library(car)
library(performance)
#https://stat.ethz.ch/~meier/teaching/anova/random-and-mixed-effects-models.html#mixed-effects-models
library(effectsize)
library(emmeans)

Loading required package: Matrix


Attaching package: 'Matrix'


The following objects are masked from 'package:tidyr':

    expand, pack, unpack



Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Loading required package: carData


Attaching package: 'car'


The following object is masked from 'package:dplyr':

    recode


Warning message:
"package 'performance' was built under R version 4.5.2"
Warning message:
"package 'effectsize' was built under R version 4.5.2"
Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'



In [5]:
### Define model formula
ModelAnalysis <- lmer(
  Value ~ Modality + CommonTrials + (1 | DatasetID),
  data = df_model
)
### Print Summary
ampsum    <- summary(ModelAnalysis)
model_amp <- as.data.frame(round(ampsum$coefficients, 3))
ampsum

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: Value ~ Modality + CommonTrials + (1 | DatasetID)
   Data: df_model

REML criterion at convergence: -196.8

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-1.66841 -0.64957 -0.01175  0.58311  1.73347 

Random effects:
 Groups    Name        Variance  Std.Dev.
 DatasetID (Intercept) 0.0006580 0.02565 
 Residual              0.0003537 0.01881 
Number of obs: 52, groups:  DatasetID, 26

Fixed effects:
               Estimate Std. Error         df t value Pr(>|t|)  
(Intercept)   0.1528164  0.0652024 24.0769208   2.344   0.0277 *
ModalityOPM  -0.0128777  0.0052160 25.0000000  -2.469   0.0207 *
CommonTrials -0.0001484  0.0003084 23.9999998  -0.481   0.6348  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Correlation of Fixed Effects:
            (Intr) MdlOPM
ModalityOPM -0.040       
CommonTrils -0.995  0.000

In [6]:
### Print Analysis of Fixed Effects
print("Fixed effects analysis")
aovamp <- anova(ModelAnalysis, type="II", test="F")
print(aovamp)
eff   <- eta_squared(aovamp, alternative="two.sided")
round(eff$Eta2_partial, 3)
round(eff$CI_low, 3)
round(eff$CI_high, 3)

aovamp <- round(as.data.frame(aovamp), 3)
aovamp

[1] "Fixed effects analysis"
Type II Analysis of Variance Table with Satterthwaite's method
                 Sum Sq    Mean Sq NumDF DenDF F value  Pr(>F)  
Modality     0.00215585 0.00215585     1    25  6.0953 0.02074 *
CommonTrials 0.00008187 0.00008187     1    24  0.2315 0.63478  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


[1] 0.196 0.010

[1] 0.003 0.000

[1] 0.450 0.195

,Sum Sq,Mean Sq,NumDF,DenDF,F value,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Modality,0.002,0.002,1,25,6.095,0.021
CommonTrials,0.000,0.000,1,24,0.231,0.635


In [7]:
### Print Effect Size
df_wide <- df_model |>
  select(DatasetID, Modality, Value) |>
  pivot_wider(names_from = Modality, values_from = Value)

### Caculate Cohen's d and Hedges' g
D   <- df_wide$OPM - df_wide$EEG
dz  <- mean(D) / sd(D)
n   <- length(D)
g_z <- dz * (1 - 3/(4*n - 9))
print("Hedges' g")
round(g_z, 3)
print("Cohen's d")
round(dz, 3)

### Calculate Probability of OPM Superiority
n_total         <- nrow(df_wide)
n_opm_better    <- sum(df_wide$OPM > df_wide$EEG)
cles            <- (n_opm_better / n_total) * 100

print("Probability of OPM superiority")
round(cles, 1)

[1] "Hedges' g"


[1] -0.469

[1] "Cohen's d"


[1] -0.484

[1] "Probability of OPM superiority"


[1] 30.8

In [ ]:
df_model